# RedQueen — GDGoC AI Challenge 2026 | Kaggle Training Notebook

**Pipeline:**
1. Setup — install deps, clone repo  
2. Phase 0 — History mining (optional, if `history_game/` dataset is attached)  
3. Phase 2 — Behavioral Cloning  
4. Phase 3 — PPO + Curriculum  
5. ONNX Export  
6. Prepare submission folder (3 files: `agent.py`, `model.onnx`, `requirements.txt`)  
7. **Last cell** — Zip both output folders for download

**Outputs:**
- `/kaggle/working/training_artifacts/` — all checkpoints, logs, BC dataset  
- `/kaggle/working/submission/` — competition-ready 3-file folder  
- `/kaggle/working/training_artifacts.zip`  
- `/kaggle/working/submission.zip` (agent.py at root ✓)

In [1]:
# ── Cell 1: Environment setup ───────────────────────────────────────────────
import os

# Suppress TF/XLA/gRPC C++ noise that fires when CUDA subprocesses start.
# Must be set BEFORE any subprocess spawns so worker processes inherit them.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["GLOG_minloglevel"]      = "3"
os.environ["GRPC_VERBOSITY"]        = "ERROR"

import warnings
# Suppress SB3 v1.8+ net_arch deprecation warning (already fixed in code, belt+suspenders)
warnings.filterwarnings("ignore", message=".*shared layers.*", category=UserWarning)

import subprocess, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "redqueen"

# Install extra dependencies not available on Kaggle by default
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "sb3-contrib>=2.3.0",
    "stable-baselines3>=2.3.0",
    "gymnasium>=0.29.1",
    "onnx>=1.16.0",
    "onnxruntime>=1.18.0",
], check=True)

# Remove old unmaintained gym if it was pre-installed by Kaggle base image
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "gym"],
               capture_output=True)

# Clone the repo (replace with your actual repo URL)
REPO_URL = "https://github.com/CryAndRRich/redqueen.git"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

# Add repo to Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Setup complete. Repo at:", REPO_DIR)
print("Python:", sys.version)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.7 MB/s eta 0:00:00


Cloning into '/kaggle/working/redqueen'...


Setup complete. Repo at: /kaggle/working/redqueen
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
# ── Cell 2: Configure output directories ────────────────────────────────────
import os
from pathlib import Path

ARTIFACTS_DIR  = WORKING / 'training_artifacts'
CKPT_DIR       = ARTIFACTS_DIR / 'checkpoints'
DATA_DIR       = ARTIFACTS_DIR / 'data'
LOGS_DIR       = ARTIFACTS_DIR / 'logs'
PAST_AGENTS_DIR= CKPT_DIR / 'past_agents'
SUBMISSION_DIR = WORKING / 'submission'

for d in [CKPT_DIR, DATA_DIR, LOGS_DIR, PAST_AGENTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Artifacts dir:', ARTIFACTS_DIR)
print('Submission dir:', SUBMISSION_DIR)

Artifacts dir: /kaggle/working/training_artifacts
Submission dir: /kaggle/working/submission


In [3]:
# ── Cell 3: Phase 0 — History mining (optional) ──────────────────────────────
#
# To use this: attach history_game/ as a Kaggle dataset input
# and update HISTORY_DIR below.
#
# If not available, this cell is skipped and we go straight to
# Phase 2 using GeniusRuleAgent self-rollout data.
#
# Output: DATA_DIR/bc_dataset/  (directory of memmap .npy files)

import os, sys
from pathlib import Path

# Look for history_game in common Kaggle input locations
HISTORY_CANDIDATES = [
    Path("/kaggle/input/datasets/nhtquyn/historygame/history_game/2026-05-23"),
    Path("/kaggle/input/bomberland-history/history_game"),
    REPO_DIR / "history_game",
]
HISTORY_DIR = next((p for p in HISTORY_CANDIDATES if p.exists()), None)

BC_DATASET = DATA_DIR / "bc_dataset"   # directory with .npy memmap files
USE_HISTORY_BC = HISTORY_DIR is not None and not (BC_DATASET / "_n.npy").exists()

if USE_HISTORY_BC:
    print(f"Found history_game at {HISTORY_DIR}")
    n_files = sum(1 for _ in HISTORY_DIR.rglob("*.json"))
    print(f"Match files: {n_files:,}")

    from src.training.history_parser import parse_history
    parse_history(
        history_dir=HISTORY_DIR,
        output_path=BC_DATASET,
        max_files=None,      # use all
        min_survival=120,
        min_bombs=5,
    )
elif (BC_DATASET / "_n.npy").exists():
    n = int(__import__("numpy").load(BC_DATASET / "_n.npy"))
    print(f"BC dataset already exists: {BC_DATASET}  ({n:,} transitions)")
else:
    print("No history_game found → will generate BC data from GeniusRuleAgent rollouts (Cell 3b)")

Found history_game at /kaggle/input/datasets/nhtquyn/historygame/history_game/2026-05-23
Match files: 2,436
Found 2,436 JSON files in /kaggle/input/datasets/nhtquyn/historygame/history_game/2026-05-23


Pass 1/2 — counting transitions: 100%|██████████| 2436/2436 [00:32<00:00, 74.45it/s]


Pass 1 complete: 2192 quality trajectories, 969,367 transitions
Pre-allocated 9.87 GB on disk at /kaggle/working/training_artifacts/data/bc_dataset/


Pass 2/2 — extracting features: 100%|██████████| 2192/2192 [03:23<00:00, 10.76it/s]



Saved dataset → /kaggle/working/training_artifacts/data/bc_dataset/  (969,367 samples)
  Files: spatial.npy, aux.npy, actions.npy, action_masks.npy, _n.npy
  Skipped 0 files (parse errors)
Action distribution:
  STOP  (0):  106432  11.0%
  LEFT  (1):  228533  23.6%
  RIGHT (2):  224974  23.2%
  UP    (3):  160651  16.6%
  DOWN  (4):  157734  16.3%
  BOMB  (5):   91043  9.4%


In [4]:
# ── Cell 3b: Generate BC data via GeniusRuleAgent self-rollout ───────────────
# (runs only if history_game/ not available)
#
# Streams to disk in chunks — avoids accumulating 10GB+ in RAM.
# Output: DATA_DIR/bc_dataset/  (same format as Cell 3)

import numpy as np
import shutil
from pathlib import Path

BC_DATASET = DATA_DIR / "bc_dataset"

if (BC_DATASET / "_n.npy").exists():
    n = int(np.load(BC_DATASET / "_n.npy"))
    print(f"BC dataset exists ({BC_DATASET}, {n:,} transitions), skipping rollout generation")
else:
    print("Generating BC dataset from GeniusRuleAgent rollouts...")
    import sys
    sys.path.insert(0, str(REPO_DIR))

    from engine.game import BomberEnv
    from agent import GeniusRuleAgent
    from src.utils.feature_extractor import extract_features, count_boxes
    from src.logic.action_masking import compute_action_mask

    N_GAMES = 5_000   # ← tune: 5k ≈ 650k transitions ≈ 6.5 GB
    MAX_STEPS = 500
    CHUNK_SIZE = 50_000   # transitions per chunk file — each chunk ≈ 500 MB

    TMP_DIR = DATA_DIR / "_rollout_chunks"
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    chunk_idx = 0
    sp_buf, aux_buf, act_buf, mask_buf = [], [], [], []

    def flush_chunk():
        global chunk_idx, sp_buf, aux_buf, act_buf, mask_buf
        if not act_buf:
            return
        np.save(TMP_DIR / f"sp_{chunk_idx:04d}.npy",   np.stack(sp_buf).astype(np.float32))
        np.save(TMP_DIR / f"aux_{chunk_idx:04d}.npy",  np.stack(aux_buf).astype(np.float32))
        np.save(TMP_DIR / f"act_{chunk_idx:04d}.npy",  np.array(act_buf, dtype=np.int64))
        np.save(TMP_DIR / f"mask_{chunk_idx:04d}.npy", np.stack(mask_buf).astype(bool))
        chunk_idx += 1
        sp_buf, aux_buf, act_buf, mask_buf = [], [], [], []

    for game_seed in range(N_GAMES):
        env = BomberEnv(max_steps=MAX_STEPS, seed=game_seed)
        agents = [GeniusRuleAgent(i) for i in range(4)]
        obs = env.reset(seed=game_seed)
        initial_boxes = count_boxes(obs["map"])
        step = 0

        while True:
            actions_taken = [int(a.act(obs)) for a in agents]

            for aid in range(4):
                if int(obs["players"][aid][2]) == 0:
                    continue
                sp, aux = extract_features(obs, aid, step=step, initial_boxes=initial_boxes)
                mask = compute_action_mask(obs, aid)
                sp_buf.append(sp)
                aux_buf.append(aux)
                act_buf.append(actions_taken[aid])
                mask_buf.append(mask)

            if len(act_buf) >= CHUNK_SIZE:
                flush_chunk()

            next_obs, terminated, truncated = env.step(actions_taken)
            obs = next_obs
            step += 1
            if terminated or truncated:
                break

        if (game_seed + 1) % 500 == 0:
            print(f"  Game {game_seed+1}/{N_GAMES} | buffered {len(act_buf):,} | chunks {chunk_idx}")

    flush_chunk()  # final partial chunk

    # ── Merge chunks into memmap directory ──────────────────────────────── #
    chunk_counts = [len(np.load(TMP_DIR / f"act_{i:04d}.npy")) for i in range(chunk_idx)]
    n_total = sum(chunk_counts)
    print(f"Merging {chunk_idx} chunks → {n_total:,} transitions")

    BC_DATASET.mkdir(parents=True, exist_ok=True)
    sp_mm   = np.memmap(BC_DATASET / "spatial.npy",      dtype="float32", mode="w+", shape=(n_total, 15, 13, 13))
    aux_mm  = np.memmap(BC_DATASET / "aux.npy",          dtype="float32", mode="w+", shape=(n_total, 7))
    act_mm  = np.memmap(BC_DATASET / "actions.npy",      dtype="int64",   mode="w+", shape=(n_total,))
    mask_mm = np.memmap(BC_DATASET / "action_masks.npy", dtype="bool",    mode="w+", shape=(n_total, 6))

    offset = 0
    for i in range(chunk_idx):
        sp_c   = np.load(TMP_DIR / f"sp_{i:04d}.npy")
        aux_c  = np.load(TMP_DIR / f"aux_{i:04d}.npy")
        act_c  = np.load(TMP_DIR / f"act_{i:04d}.npy")
        mask_c = np.load(TMP_DIR / f"mask_{i:04d}.npy")
        n_c = len(act_c)
        sp_mm[offset:offset+n_c]   = sp_c
        aux_mm[offset:offset+n_c]  = aux_c
        act_mm[offset:offset+n_c]  = act_c
        mask_mm[offset:offset+n_c] = mask_c
        offset += n_c

    del sp_mm, aux_mm, act_mm, mask_mm   # flush to disk
    np.save(BC_DATASET / "_n.npy", np.array(n_total, dtype=np.int64))
    shutil.rmtree(TMP_DIR)
    print(f"Saved BC dataset → {BC_DATASET}/  ({n_total:,} transitions)")

BC dataset exists (/kaggle/working/training_artifacts/data/bc_dataset, 969,367 transitions), skipping rollout generation


In [5]:
# ── Cell 4: Phase 2 — Behavioral Cloning ────────────────────────────────────

from pathlib import Path
import sys
sys.path.insert(0, str(REPO_DIR))

from src.training.bc_trainer import train_bc

BC_DATASET = DATA_DIR / "bc_dataset"   # directory with memmap .npy files
assert (BC_DATASET / "_n.npy").exists(), f"BC dataset not found: {BC_DATASET}"

# ~6 min per 10 epochs on Kaggle T4 → 40 epochs ≈ 25 min
BC_EPOCHS = 40

bc_best_ckpt = train_bc(
    dataset_path=BC_DATASET,
    output_dir=CKPT_DIR,
    epochs=BC_EPOCHS,
    batch_size=512,
    lr=3e-4,
    gamma_focal=2.0,
    device="auto",
    save_every=10,
)

print(f"BC best checkpoint: {bc_best_ckpt}")

E0000 00:00:1779599222.300346      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779599222.355199      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779599222.851297      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779599222.851339      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779599222.851342      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779599222.851345      22 computation_placer.cc:177] computation placer already registered. Please check linka

Device: cuda
Loading dataset from /kaggle/working/training_artifacts/data/bc_dataset …
Dataset: 969,367 transitions  (memmap — lazy)


Epoch   1 | train loss 122646.4437 acc 0.404 | val loss 82529.1825 acc 0.499 | lr 3.00e-04


Epoch   2 | train loss 122646.2634 acc 0.529 | val loss 82529.1092 acc 0.550 | lr 2.98e-04


Epoch   3 | train loss 122646.2077 acc 0.568 | val loss 82529.0737 acc 0.575 | lr 2.96e-04


...

Epoch  36 | train loss 122645.8804 acc 0.758 | val loss 82529.1256 acc 0.640 | lr 7.34e-06


Epoch  37 | train loss 122645.8789 acc 0.759 | val loss 82529.1311 acc 0.639 | lr 4.14e-06


Epoch  38 | train loss 122645.8780 acc 0.760 | val loss 82529.1313 acc 0.639 | lr 1.85e-06


Epoch  39 | train loss 122645.8775 acc 0.761 | val loss 82529.1332 acc 0.639 | lr 4.62e-07


Epoch  40 | train loss 122645.8768 acc 0.762 | val loss 82529.1339 acc 0.639 | lr 0.00e+00
  → Saved bc_40ep_20260524_052910.pt

Best val loss: 82528.9813 → /kaggle/working/training_artifacts/checkpoints/bc_best.pt
BC best checkpoint: /kaggle/working/training_artifacts/checkpoints/bc_best.pt


In [6]:
# ── Cell 5: Phase 3 — PPO curriculum training ────────────────────────────────
#
# Budget guide (Kaggle T4, N_ENVS=4):
#   BC 40 epochs           ≈  25 min
#   PPO 4 stages × 500k   ≈ 160 min  (may advance earlier if win rate is high)
#   Total                  ≈  ~3–4 h
#
# Each stage trains up to PPO_STEPS_PER_STAGE but advances early when
# win_rate >= threshold for 3 consecutive eval windows (every 50k steps).

import sys
sys.path.insert(0, str(REPO_DIR))

from src.training.ppo_trainer import train_curriculum

# ← Tune for time budget. 100k ≈ 3–5 min/stage at N_ENVS=4.
PPO_STEPS_PER_STAGE = 500_000
N_ENVS = 4   # increase to 8 for ~1.5× speed if VRAM allows

ppo_best_ckpt = train_curriculum(
    output_dir=CKPT_DIR,
    total_steps_per_stage=PPO_STEPS_PER_STAGE,
    n_envs=N_ENVS,
    init_from=bc_best_ckpt,
    device='auto',
)

print(f'PPO curriculum best: {ppo_best_ckpt}')


=== Curriculum Stage 0: random (threshold 60%) ===


E0000 00:00:1779600557.000123     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779600557.000122     114 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779600557.013187     114 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1779600557.013187     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779600557.046508     114 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779600557.046508     113 computation_placer.cc:177] computation placer already registered. Please check li

Using cuda device
Loaded BC weights from bc_best.pt
Logging to /kaggle/working/training_artifacts/checkpoints/tb_logs/stage_0_random_0
-----------------------------
| time/              |      |
|    fps             | 800  |
|    iterations      | 1    |
|    time_elapsed    | 10   |
|    total_timesteps | 8192 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 689         |
|    iterations           | 2           |
|    time_elapsed         | 23          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.008395284 |
|    clip_fraction        | 0.0778      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.05       |
|    explained_variance   | -0.00227    |
|    learning_rate        | 0.0003      |
|    loss                 | 0.201       |
|    n_updates            | 10          |
|    policy_gradient_loss

E0000 00:00:1779600872.094887     191 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779600872.094885     189 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779600872.106645     189 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1779600872.106917     191 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779600872.136486     189 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779600872.136486     191 computation_placer.cc:177] computation placer already registered. Please check li

Logging to /kaggle/working/training_artifacts/checkpoints/tb_logs/stage_1_simple_0
-------------------------------
| time/              |        |
|    fps             | 746    |
|    iterations      | 1      |
|    time_elapsed    | 10     |
|    total_timesteps | 180224 |
-------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 653        |
|    iterations           | 2          |
|    time_elapsed         | 25         |
|    total_timesteps      | 188416     |
| train/                  |            |
|    approx_kl            | 0.05615627 |
|    clip_fraction        | 0.291      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.591     |
|    explained_variance   | 0.64       |
|    learning_rate        | 0.0003     |
|    loss                 | -0.0524    |
|    n_updates            | 220        |
|    policy_gradient_loss | -0.0374    |
|    value_loss           | 0.262    

E0000 00:00:1779602404.870348     271 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779602404.883595     271 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779602404.919246     271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779602404.919289     271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779602404.919295     271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779602404.919299     271 computation_placer.cc:177] computation placer already registered. Please check linka

Logging to /kaggle/working/training_artifacts/checkpoints/tb_logs/stage_2_genius_nobomb_0
-------------------------------
| time/              |        |
|    fps             | 734    |
|    iterations      | 1      |
|    time_elapsed    | 11     |
|    total_timesteps | 753664 |
-------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 648        |
|    iterations           | 2          |
|    time_elapsed         | 25         |
|    total_timesteps      | 761856     |
| train/                  |            |
|    approx_kl            | 0.15216646 |
|    clip_fraction        | 0.354      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.43      |
|    explained_variance   | 0.92       |
|    learning_rate        | 0.0003     |
|    loss                 | -0.0445    |
|    n_updates            | 920        |
|    policy_gradient_loss | -0.0219    |
|    value_loss           | 0.

E0000 00:00:1779604687.753427     353 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779604687.765469     353 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779604687.795728     353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779604687.795764     353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779604687.795769     353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779604687.795772     353 computation_placer.cc:177] computation placer already registered. Please check linka

Logging to /kaggle/working/training_artifacts/checkpoints/tb_logs/stage_3_genius_0
--------------------------------
| time/              |         |
|    fps             | 751     |
|    iterations      | 1       |
|    time_elapsed    | 10      |
|    total_timesteps | 1327104 |
--------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 655        |
|    iterations           | 2          |
|    time_elapsed         | 24         |
|    total_timesteps      | 1335296    |
| train/                  |            |
|    approx_kl            | 0.10775616 |
|    clip_fraction        | 0.268      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.414     |
|    explained_variance   | 0.66       |
|    learning_rate        | 0.0003     |
|    loss                 | 0.038      |
|    n_updates            | 1620       |
|    policy_gradient_loss | -0.0391    |
|    value_loss           | 0.

In [7]:
# ── Cell 6 (optional): Phase 4 — Self-play ───────────────────────────────────
# Skip if time is tight. Only run if Phase 3 converged.
# ~500k steps ≈ 50 min on Kaggle T4 at N_ENVS=4.

RUN_SELF_PLAY = True   # ← set True to enable

if RUN_SELF_PLAY:
    from src.training.ppo_trainer import train_self_play

    ppo_best_ckpt = train_self_play(
        output_dir=CKPT_DIR,
        snapshot_dir=PAST_AGENTS_DIR,
        total_steps=500_000,
        n_envs=N_ENVS,
        init_from=ppo_best_ckpt,
        device='auto',
    )
    print(f'Self-play best: {ppo_best_ckpt}')
else:
    print('Self-play skipped')

E0000 00:00:1779606040.134904     436 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779606040.134885     435 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779606040.134884     437 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779606040.134902     434 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779606040.148155     435 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1779606040.148168     437 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory

Using cuda device
Loaded BC weights from ppo_curriculum_best.pt
Logging to /kaggle/working/training_artifacts/checkpoints/tb_logs/MaskablePPO_0
-----------------------------
| time/              |      |
|    fps             | 761  |
|    iterations      | 1    |
|    time_elapsed    | 10   |
|    total_timesteps | 8192 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 660          |
|    iterations           | 2            |
|    time_elapsed         | 24           |
|    total_timesteps      | 16384        |
| train/                  |              |
|    approx_kl            | 0.0039012316 |
|    clip_fraction        | 0.0399       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.347       |
|    explained_variance   | 0.0232       |
|    learning_rate        | 0.0003       |
|    loss                 | 0.00979      |
|    n_updates            | 10           |
|

In [8]:
# ── Cell 7: ONNX export ──────────────────────────────────────────────────────

import sys
sys.path.insert(0, str(REPO_DIR))

from src.utils.export_onnx import export_to_onnx

# Use the best available checkpoint
best_ckpt = ppo_best_ckpt if 'ppo_best_ckpt' in dir() else bc_best_ckpt
assert best_ckpt.exists(), f'No checkpoint found at {best_ckpt}'

onnx_path = CKPT_DIR / 'model.onnx'

export_to_onnx(
    checkpoint_path=best_ckpt,
    output_path=onnx_path,
    opset=17,
    verify=True,
)

print(f'ONNX model: {onnx_path}')

Loading checkpoint: /kaggle/working/training_artifacts/checkpoints/selfplay_best.pt
Exporting to /kaggle/working/training_artifacts/checkpoints/model.onnx (opset 17) …
Exported: /kaggle/working/training_artifacts/checkpoints/model.onnx  (3542.6 KB)
ONNX model check: OK
Max abs diff (PyTorch vs ONNX): 0.000001
Verification PASSED ✓
Inference speed (1000 runs):  median 0.21 ms  p95 0.27 ms
Inference budget check: OK (< 100 ms)
ONNX model: /kaggle/working/training_artifacts/checkpoints/model.onnx


In [9]:
# ── Cell 8: Prepare submission folder (3 files) ──────────────────────────────
#
# Competition format:
#   submission.zip
#   ├── agent.py        ← at root, MANDATORY
#   ├── model.onnx
#   └── requirements.txt

import shutil

# 1. agent.py — copy from repo
shutil.copy2(REPO_DIR / 'agent' / 'agent.py', SUBMISSION_DIR / 'agent.py')

# 2. model.onnx — copy from ONNX export
shutil.copy2(onnx_path, SUBMISSION_DIR / 'model.onnx')

# 3. requirements.txt — minimal runtime deps (CPU-only)
(SUBMISSION_DIR / 'requirements.txt').write_text(
    'numpy>=1.26.0\n'
    'onnxruntime>=1.18.0\n'
)

# Validate
sub_files = list(SUBMISSION_DIR.iterdir())
print('Submission folder contents:')
for f in sorted(sub_files):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s}  {size_kb:.1f} KB')

# Safety check: agent.py must be at root (not inside a subfolder)
assert (SUBMISSION_DIR / 'agent.py').exists(), 'FATAL: agent.py missing from submission root!'
print('\nagent.py at root: ✓')

Submission folder contents:
  agent.py                        14.2 KB
  model.onnx                      3542.6 KB
  requirements.txt                0.0 KB

agent.py at root: ✓


In [10]:
# ── Cell 9 (LAST): Zip both output folders ───────────────────────────────────
#
# After this cell completes:
#   /kaggle/working/training_artifacts.zip  — all checkpoints, logs, dataset
#   /kaggle/working/submission.zip          — 3-file competition submission
#
# Download them from the "Output" tab on the right panel.

import zipfile, os
from pathlib import Path

def zip_directory(source_dir: Path, output_zip: Path) -> None:
    """Zip source_dir into output_zip with files at root level."""
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(source_dir.rglob('*')):
            if file_path.is_file():
                arcname = file_path.relative_to(source_dir)
                zf.write(file_path, arcname)
    size_mb = output_zip.stat().st_size / (1024 * 1024)
    print(f'Created: {output_zip.name}  ({size_mb:.1f} MB)')


# ── Zip 1: training_artifacts ────────────────────────────────────────────── #
artifacts_zip = WORKING / 'training_artifacts.zip'
zip_directory(ARTIFACTS_DIR, artifacts_zip)

# ── Zip 2: submission (agent.py at root — competition format) ─────────────── #
submission_zip = WORKING / 'submission.zip'
zip_directory(SUBMISSION_DIR, submission_zip)

# ── Verify submission zip has agent.py at root ────────────────────────────── #
with zipfile.ZipFile(submission_zip, 'r') as zf:
    names = zf.namelist()

print('\nFiles in submission.zip:')
for n in sorted(names):
    print(f'  {n}')

assert 'agent.py' in names, 'FATAL: agent.py not at root of submission.zip!'
print('\nagent.py at zip root: ✓')
print('\n=== Done! Download files from the Output tab ===')
print(f'  → training_artifacts.zip  (all checkpoints)')
print(f'  → submission.zip          (upload this to the competition)')

Created: training_artifacts.zip  (273.8 MB)
Created: submission.zip  (3.1 MB)

Files in submission.zip:
  agent.py
  model.onnx
  requirements.txt

agent.py at zip root: ✓

=== Done! Download files from the Output tab ===
  → training_artifacts.zip  (all checkpoints)
  → submission.zip          (upload this to the competition)
